# Tutorial 1: Getting Started with Scalable

## What You Will Learn

By the end of this notebook you will:

- Install Scalable and verify its dependencies
- Create a minimal `scalable.yaml` manifest
- Validate, plan, and execute a local workflow end-to-end
- Inspect the telemetry output of a successful run

## Prerequisites

- Python 3.11+
- `pip install scalable` (run the cell below if not already installed)

In [ ]:
# Install Scalable (skip if already installed)
# !pip install scalable

In [1]:
# Verify installation
import scalable
print(f"Scalable version: {scalable.__version__}")

Scalable version: 2.0.0


## Step 1: Create a Project Directory and Manifest

The manifest (`scalable.yaml`) is the declarative single source of truth for your workflow.
It describes:
- **Project** metadata
- **Targets** (execution environments)
- **Components** (resource profiles for workloads)
- **Tasks** (named work units bound to components)

In [2]:
import os
import tempfile

# Create a temporary project directory for this tutorial
project_dir = tempfile.mkdtemp(prefix="scalable-tutorial-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

Working directory: /var/folders/8c/787g_pzs7wq7hljnnx3prdzr0000gn/T/scalable-tutorial-fz6cyspk


In [3]:
# Write the manifest
manifest_content = """\
version: 1
project:
  name: hello-scalable

targets:
  local:
    provider: local
    max_workers: 2
    threads_per_worker: 1
    processes: false
    containers: none

components:
  analysis:
    cpus: 1
    memory: 1G

tasks:
  run_analysis:
    component: analysis
"""

with open("scalable.yaml", "w") as f:
    f.write(manifest_content)

print("Manifest written to scalable.yaml")
print("---")
print(manifest_content)

Manifest written to scalable.yaml
---
version: 1
project:
  name: hello-scalable

targets:
  local:
    provider: local
    max_workers: 2
    threads_per_worker: 1
    processes: false
    containers: none

components:
  analysis:
    cpus: 1
    memory: 1G

tasks:
  run_analysis:
    component: analysis



### Understanding the Manifest

| Section | Purpose |
|---------|--------|
| `version: 1` | Schema version (only `1` currently supported) |
| `project.name` | Identifies the project in telemetry and artifacts |
| `targets.local` | Uses the built-in `LocalProvider` (Dask LocalCluster) |
| `components.analysis` | Declares 1 CPU / 1 GB resource profile |
| `tasks.run_analysis` | Binds a work unit to the `analysis` component |

> **Trade-off:** `processes: false` runs Dask workers as threads (fast startup, no serialization overhead) but provides no memory isolation between tasks.

## Step 2: Validate the Manifest

Validation checks structural and semantic correctness before running anything.

In [4]:
from scalable import ScalableSession

session = ScalableSession.from_yaml("./scalable.yaml", target="local")
report = session.validate()

if report.ok:
    print("✓ Manifest is valid (0 errors, 0 warnings)")
else:
    for issue in report.errors:
        print(f"ERROR [{issue.code}] {issue.path}: {issue.message}")
    for issue in report.warnings:
        print(f"WARN  [{issue.code}] {issue.path}: {issue.message}")

✓ Manifest is valid (0 errors, 0 warnings)


## Step 3: Plan the Execution

Planning produces a dry-run plan — a description of what *would* happen without allocating real resources.

In [5]:
plan = session.plan(dry_run=True)

print(f"Target: {plan.target_name}")
print(f"Provider: {plan.provider_name}")
print(f"Manifest lock: {plan.manifest_lock}")
print(f"Scale plan: {plan.scale_plan}")

Target: local
Provider: local
Manifest lock: 7d378b80bf6861695d621c66c8e523bd3ef735a7918eca547933e4dda7e471d2
Scale plan: ScalePlan(workers_by_tag={'analysis': 1}, resources_by_tag={'analysis': ResourceRequest(cpus=1, memory='1G', walltime=None, gpus=None)})


The `manifest_lock` is a content-addressable hash. Two plans with the same lock were derived from byte-identical configurations — this guarantees reproducibility.

## Step 4: Write and Run a Workflow

Now let's define a simple computation and execute it through Scalable.

In [6]:
import time


def analyze(scenario_id: int) -> dict:
    """Simulate an expensive computation."""
    time.sleep(0.5)  # Shortened for notebook
    return {"scenario": scenario_id, "result": scenario_id * 42}


# Start the cluster
client = session.start(plan)
print(f"Client connected: {client}")

Client connected: <ScalableClient: 'inproc://192.168.1.212/58145/1' processes=1 threads=1, memory=64.00 GiB>


In [7]:
# Submit tasks tagged to the 'analysis' component
futures = []
for i in range(5):
    fut = client.submit(analyze, i, tag="analysis")
    futures.append(fut)

print(f"Submitted {len(futures)} tasks")

Submitted 5 tasks


In [8]:
# Gather results
results = client.gather(futures)

for r in results:
    print(r)

{'scenario': 0, 'result': 0}
{'scenario': 1, 'result': 42}
{'scenario': 2, 'result': 84}
{'scenario': 3, 'result': 126}
{'scenario': 4, 'result': 168}


### What Happened Under the Hood

1. `ScalableSession.from_yaml` parsed the manifest and built a `DeploymentSpec`
2. `session.plan()` validated and computed resource allocation (2 workers × 1 CPU / 1G)
3. `session.start()` created a Dask `LocalCluster` with the specified workers
4. Each `client.submit(..., tag="analysis")` routed the function to matching workers
5. Results were gathered back to the client

## Step 5: Close the Session and Inspect Telemetry

In [9]:
# Always close the session to finalize telemetry
session.close()
print("Session closed.")

Session closed.


In [10]:
# Check telemetry output
from pathlib import Path

runs_dir = Path(".scalable/runs")
if runs_dir.exists():
    run_dirs = sorted(runs_dir.iterdir())
    if run_dirs:
        latest_run = run_dirs[-1]
        print(f"Latest run: {latest_run.name}")
        print(f"\nContents:")
        for f in sorted(latest_run.iterdir()):
            size = f.stat().st_size
            print(f"  {f.name} ({size} bytes)")
else:
    print("No telemetry data found (telemetry may be disabled)")

Latest run: run-20260520T125819Z-hello-scalable-1143230e

Contents:
  manifest.lock (65 bytes)
  manifest.yaml (266 bytes)
  plan.json (435 bytes)
  resources.jsonl (2050 bytes)
  run.json (412 bytes)
  summary.json (1153 bytes)
  tasks.jsonl (6354 bytes)
  workers.jsonl (832 bytes)


In [11]:
# Read run metadata
import json

if runs_dir.exists() and run_dirs:
    run_json = latest_run / "run.json"
    if run_json.exists():
        with open(run_json) as f:
            meta = json.load(f)
        print("Run metadata:")
        for key, value in meta.items():
            print(f"  {key}: {value}")

Run metadata:
  finished_at: 2026-05-20T12:58:53Z
  manifest_lock: 7d378b80bf6861695d621c66c8e523bd3ef735a7918eca547933e4dda7e471d2
  project_name: hello-scalable
  provider_name: local
  run_id: run-20260520T125819Z-hello-scalable-1143230e
  schema_version: 1
  source_manifest_path: scalable.yaml
  started_at: 2026-05-20T12:58:19Z
  status: completed
  target_name: local


## Step 6: Environment Variables

Scalable is configured through environment variables for deployment flexibility:

In [12]:
from scalable.common import settings

print("Current settings:")
print(f"  Cache dir: {settings.cache_dir}")
print(f"  Seed: {settings.seed}")
print(f"  Manifest path: {settings.manifest_path}")
print(f"  Runs dir: {settings.runs_dir}")
print(f"  Telemetry enabled: {settings.telemetry_enabled}")

Current settings:
  Cache dir: ./cache
  Seed: 987654321
  Manifest path: ./scalable.yaml
  Runs dir: ./.scalable/runs
  Telemetry enabled: True


## Summary

In this tutorial you:
1. Created a minimal `scalable.yaml` manifest
2. Validated it for structural correctness
3. Generated a dry-run execution plan
4. Ran a workflow through the Session API
5. Inspected telemetry output

## Next Steps

- **Tutorial 2**: Deep-dive into the manifest system (targets, overlays, env vars)
- **Tutorial 4**: Add `@cacheable` to skip redundant computation
- **Tutorial 6**: Analyze telemetry data for performance insights

In [13]:
# Cleanup
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print("Tutorial workspace cleaned up.")

Tutorial workspace cleaned up.
